# 03_indice_oportunidad

## Objetivo
Calcular el índice de oportunidad de microcrédito digital por departamento a partir del dataset maestro.

## Alcance
- Cargar `master_dataset.csv`
- Normalizar variables
- Definir pesos
- Calcular índice
- Generar ranking
- Explorar escenarios (opcional)

## 1. Librerías

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
pd.set_option('display.max_columns', None)

## 2. Rutas y carga de datos

In [2]:
PROJECT_ROOT = Path.cwd().resolve().parent
DATA_PROCESSED = PROJECT_ROOT / 'data' / 'processed'

df = pd.read_csv(DATA_PROCESSED / 'master_dataset.csv')
df.head()

,departamento,pobreza_2024,acceso_microcredito_2024,acceso_productos_financieros_2024,atm_x_10000_adultos_2024,internet_hogares_2024
0,ANTIOQUIA,24.7,4.4,126.3,4.6,66.9
1,ATLANTICO,31.6,2.8,94.2,4.5,58.9
2,BOGOTA D.C.,19.6,2.9,118.8,7.5,82.7
3,BOLIVAR,48.0,4.4,82.6,3.7,60.2
4,BOYACA,30.9,12.3,85.4,3.5,60.6


## 3. Revisión inicial

In [3]:
print(df.shape)
print(df.isna().sum())
df.describe()

(24, 6)
departamento                         0
pobreza_2024                         0
acceso_microcredito_2024             0
acceso_productos_financieros_2024    0
atm_x_10000_adultos_2024             0
internet_hogares_2024                0
dtype: int64


,pobreza_2024,acceso_microcredito_2024,acceso_productos_financieros_2024,atm_x_10000_adultos_2024,internet_hogares_2024
count,24.000000,24.000000,24.000000,24.000000,24.000000
mean,37.166667,6.704167,86.304167,3.562500,60.854167
std,14.188840,3.341502,16.372896,1.351589,13.397663
min,19.600000,2.800000,53.700000,1.400000,28.700000
25%,25.450000,4.400000,77.275000,2.575000,54.975000
50%,35.450000,5.650000,85.200000,3.600000,62.700000
75%,47.850000,8.425000,94.225000,4.500000,67.875000
max,67.400000,13.400000,126.300000,7.500000,82.700000


## 4. Normalización de variables
Se utiliza Min-Max scaling para llevar todas las variables al rango [0,1].

In [4]:
def minmax(series):
    return (series - series.min()) / (series.max() - series.min())

df_norm = df.copy()

df_norm['pobreza_n'] = minmax(df['pobreza_2024'])
df_norm['microcredito_n'] = 1 - minmax(df['acceso_microcredito_2024'])
df_norm['productos_n'] = 1 - minmax(df['acceso_productos_financieros_2024'])
df_norm['atm_n'] = 1 - minmax(df['atm_x_10000_adultos_2024'])
df_norm['internet_n'] = minmax(df['internet_hogares_2024'])

df_norm[['departamento','pobreza_n','microcredito_n','productos_n','atm_n','internet_n']].head()

,departamento,pobreza_n,microcredito_n,productos_n,atm_n,internet_n
0,ANTIOQUIA,0.106695,0.849057,0.000000,0.475410,0.707407
1,ATLANTICO,0.251046,1.000000,0.442149,0.491803,0.559259
2,BOGOTA D.C.,0.000000,0.990566,0.103306,0.000000,1.000000
3,BOLIVAR,0.594142,0.849057,0.601928,0.622951,0.583333
4,BOYACA,0.236402,0.103774,0.563361,0.655738,0.590741


## 5. Definición de pesos
Los pesos representan la importancia relativa de cada dimensión.

In [5]:
pesos = {
    'pobreza_n': 0.30,
    'microcredito_n': 0.30,
    'productos_n': 0.15,
    'atm_n': 0.10,
    'internet_n': 0.15
}

pesos

{'pobreza_n': 0.3,
 'microcredito_n': 0.3,
 'productos_n': 0.15,
 'atm_n': 0.1,
 'internet_n': 0.15}

## 6. Cálculo del índice de oportunidad

In [6]:
df_norm['indice_oportunidad'] = (
    df_norm['pobreza_n'] * pesos['pobreza_n'] +
    df_norm['microcredito_n'] * pesos['microcredito_n'] +
    df_norm['productos_n'] * pesos['productos_n'] +
    df_norm['atm_n'] * pesos['atm_n'] +
    df_norm['internet_n'] * pesos['internet_n']
)

df_norm[['departamento','indice_oportunidad']].head()

,departamento,indice_oportunidad
0,ANTIOQUIA,0.440377
1,ATLANTICO,0.574705
2,BOGOTA D.C.,0.462666
3,BOLIVAR,0.673044
4,BOYACA,0.340742


## 7. Ranking de departamentos

In [7]:
ranking = df_norm.sort_values(by='indice_oportunidad', ascending=False).reset_index(drop=True)
ranking[['departamento','indice_oportunidad']].head(10)

,departamento,indice_oportunidad
0,CHOCO,0.818868
1,LA GUAJIRA,0.809128
2,MAGDALENA,0.688781
3,SUCRE,0.678311
4,BOLIVAR,0.673044
5,CORDOBA,0.656790
6,CESAR,0.654618
7,ATLANTICO,0.574705
8,NORTE DE SANTANDER,0.551383
9,CAUCA,0.528884


## 8. Clasificación por niveles

In [8]:
df_norm['nivel'] = pd.qcut(df_norm['indice_oportunidad'], q=3, labels=['Bajo','Medio','Alto'])
df_norm[['departamento','indice_oportunidad','nivel']].sort_values(by='indice_oportunidad', ascending=False).head(10)

,departamento,indice_oportunidad,nivel
9,CHOCO,0.818868,Alto
13,LA GUAJIRA,0.809128,Alto
14,MAGDALENA,0.688781,Alto
21,SUCRE,0.678311,Alto
3,BOLIVAR,0.673044,Alto
10,CORDOBA,0.656790,Alto
8,CESAR,0.654618,Alto
1,ATLANTICO,0.574705,Alto
17,NORTE DE SANTANDER,0.551383,Medio
7,CAUCA,0.528884,Medio


## 9. Guardar resultados

In [9]:
output = DATA_PROCESSED / 'ranking_oportunidad.csv'
df_norm.to_csv(output, index=False)
print('Guardado en:', output)

Guardado en: C:\Users\guill\microAI\data\processed\ranking_oportunidad.csv


## 10. Exploración de escenarios (opcional)
Permite modificar pesos para observar cambios en el ranking.

In [10]:
# ejemplo de escenario alternativo
pesos_alt = pesos.copy()
pesos_alt['internet_n'] = 0.30
pesos_alt['pobreza_n'] = 0.20

df_norm['indice_alt'] = (
    df_norm['pobreza_n'] * pesos_alt['pobreza_n'] +
    df_norm['microcredito_n'] * pesos_alt['microcredito_n'] +
    df_norm['productos_n'] * pesos_alt['productos_n'] +
    df_norm['atm_n'] * pesos_alt['atm_n'] +
    df_norm['internet_n'] * pesos_alt['internet_n']
)

df_norm.sort_values(by='indice_alt', ascending=False)[['departamento','indice_alt']].head(10)

,departamento,indice_alt
9,CHOCO,0.718868
13,LA GUAJIRA,0.716573
3,BOLIVAR,0.701130
8,CESAR,0.695066
14,MAGDALENA,0.694960
21,SUCRE,0.648189
10,CORDOBA,0.645418
1,ATLANTICO,0.633490
23,VALLE DEL CAUCA,0.626204
19,RISARALDA,0.615858
